# Step 04 — semantic enrichment

Step 03 answered *where the buildings are and how big they are*. This notebook
answers **what each one is for**.

| | |
|---|---|
| **Reads** | `data/output/03_alkis_by_building.gpkg` (426 MB, 869,316 buildings) |
| **Writes** | `data/output/04_function_activity_review.csv` — the keep/drop decision table |
| | `data/experimental_extract/04_buildings_labelled.gpkg` + `.qml` — for QGIS |
| **Needs** | `geopandas`, `pyogrio` |

## State of this notebook

Built **one step at a time**, each run and inspected before the next is written.

| step | | status |
|---|---|---|
| **04.1** | **Slim the layer** — 30 columns → 18, addresses combined | **implemented** |
| **04.2** | **Function labels + activity map, and a review table** | **implemented** |
| **04.3** | **Drop the home-only buildings** | **implemented** |
| 04.4 | Join the OSM POI layer by `poi_role` | not written |
| 04.5 | Write the enriched layer | not written |

## Which input layer, and why

`03_alkis_by_building.gpkg` — **one row per ALKIS object**, not per LoD2 part.
Step 03 writes both and the part layer is the authoritative one, but for
semantics the building is the right unit: `function`, `name` and the address are
attributes of the ALKIS object, so every part of a building carries identical
values. Measured in step 03: **0 of 869,316 buildings have parts that disagree
on `function`.** Nothing semantic is lost by working at building level, and a
POI sitting inside a hospital belongs to the hospital rather than to whichever
wing happens to contain it.

The part layer is there to fall back on if a building-level answer ever looks
wrong.

In [ ]:
import os, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
# Same reason as steps 01-03: a notebook's working directory is not necessarily
# its own folder, so Path('..') is unreliable.
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError(
        'Cannot find the pipeline root (the folder containing config.py). '
        f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.'
    )
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import time
import numpy as np
import pandas as pd
import geopandas as gpd

from config import (
    ALKIS_BY_BUILDING_FILE, ALKIS_SLIM_COLS, ALKIS_ADDRESS_PARTS, TARGET_CRS,
    BUILDING_FUNCTION_CODELIST_FILE, ALKIS_ACTIVITY_MAP_FILE, ALKIS_ACTIVITY_SEP,
    FUNCTION_REVIEW_CSV, LABELLED_INSPECT_FILE, EXPERIMENTAL_DIR, OUTPUT_DIR,
    ALKIS_HOME_ONLY_ACTIVITIES, ALKIS_EXPECTED_HOME_ONLY,
)
from lib.checks import require_file, require_non_empty, require_crs, require_unique

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

print('Root :', ROOT_DIR)
print('Input:', ALKIS_BY_BUILDING_FILE.name)

## 1. Input contract

The building layer must be what step 03 promised: one row per `alkis_id`, flat
2D geometry in the target CRS, with `function` populated everywhere.

`function` being 100 % filled is the one that matters — it is the parameter the
whole classification turns on, and a gap there would propagate silently into
every activity assignment.

In [ ]:
require_file(ALKIS_BY_BUILDING_FILE, 'ALKIS buildings (step 03)')

print('Reading (~20 s) ...', flush=True)
t0 = time.perf_counter()
bld = gpd.read_file(ALKIS_BY_BUILDING_FILE, layer='buildings')
print(f'  ok  {len(bld):,} rows x {len(bld.columns)} columns  '
      f'[{time.perf_counter() - t0:,.0f}s]')

require_non_empty(bld, 'buildings')
require_crs(bld, TARGET_CRS, 'buildings')
require_unique(bld, 'alkis_id', 'buildings')

if bld.geometry.has_z.any():
    raise AssertionError('geometry still carries Z - step 03 should have flattened it')
print(f'  ok  geometry is 2D: {bld.geometry.geom_type.value_counts().to_dict()}')

_nf = int(bld['function'].isna().sum())
if _nf:
    raise AssertionError(
        f'{_nf:,} buildings have no `function` code. Everything in this notebook '
        'keys off it, so a gap here would propagate silently.'
    )
print(f'  ok  function: 100 % filled, {bld["function"].nunique()} distinct codes')

## 2. What arrived

Two things worth looking at before dropping anything.

**The function codes.** These are AdV `Gebaeudefunktion` values in the form
`<AAA object class>_<function>`. The prefix matters: `31001` is `AX_Gebaeude`,
an actual building, and the `51xxx` classes are other structures — canopies,
installations. Step 03 measured **11.57 % of rows are not `31001`**, which is
the single most important thing to know before joining the reference tables in
04.2: if they only cover `31001_*`, roughly 100,000 rows fall out of the join
with no error raised.

**Fill rates.** `name` and the address columns are sparse, and how sparse
decides what is possible later — an OSM name match can only reach the buildings
that have a name.

In [ ]:
print('AAA object class prefixes:')
_pref = bld['function'].str.slice(0, 5).value_counts()
for p, c in _pref.items():
    print(f'  {p}   {c:>9,}   ({100 * c / len(bld):>5.2f} %)')
print(f'  -> not 31001: {len(bld) - _pref.get("31001", 0):,} '
      f'({100 * (1 - _pref.get("31001", 0) / len(bld)):.2f} %)')

print()
print('the 12 most common function codes:')
for f, c in bld['function'].value_counts().head(12).items():
    print(f'  {f:<14} {c:>9,}  ({100 * c / len(bld):>5.2f} %)')

print()
print('fill rates of the columns that are about to be kept or dropped:')
for c in sorted(bld.columns):
    if c == 'geometry':
        continue
    nn = bld[c].notna().sum()
    keep = 'keep' if c in ALKIS_SLIM_COLS else ('-> address' if c in ALKIS_ADDRESS_PARTS else 'DROP')
    print(f'  {c:<20} {100 * nn / len(bld):>6.2f} %  {str(bld[c].dtype):<8}  {keep}')

## 3. Combine the address, then slim to 18 columns

`street` and `house_number` become one `address` field — `"Aachener Straße 12"`
— falling back to the street alone when the number is missing, and to NULL when
neither exists.

**Expect it empty on about half the layer.** `street` and `house_number` are
~50 % filled, so anything downstream that wants address matching can only ever
reach half the building stock. That is a property of ALKIS, not of this step,
and it is better known now than discovered during a join.

What goes, and why, is recorded in `ALKIS_SLIM_COLS` in `config.py`. The two
worth restating here:

* **`n_functions` / `functions_all`** — always `1` and always NULL, because
  `function` belongs to the ALKIS object. They were guards for a disagreement
  that cannot occur in this region.
* **`elev_ground_min_m` / `elev_top_max_m`** — sea-level elevations. Capacity
  depends on heights above ground, which are kept. Note this drops the only
  topography signal in the layer; restore `elev_ground_min_m` if a later step
  wants terrain.

In [ ]:
_street, _house = ALKIS_ADDRESS_PARTS
_s = bld[_street].fillna('').astype(str).str.strip()
_h = bld[_house].fillna('').astype(str).str.strip()
addr = (_s + ' ' + _h).str.strip()
bld['address'] = addr.where(addr != '', None)

print(f'  ..  {_street:<13} {100 * bld[_street].notna().mean():>6.2f} % filled')
print(f'  ..  {_house:<13} {100 * bld[_house].notna().mean():>6.2f} % filled')
print(f'  ..  address       {100 * bld["address"].notna().mean():>6.2f} % filled')
_street_only = int((bld[_street].notna() & bld[_house].isna()).sum())
print(f'  ..  street but no number: {_street_only:,}')
print()
print('  examples:')
for v in bld.loc[bld['address'].notna(), 'address'].head(5):
    print(f'      {v}')

missing = [c for c in ALKIS_SLIM_COLS if c not in bld.columns]
if missing:
    raise AssertionError(f'ALKIS_SLIM_COLS names columns that do not exist: {missing}')
dropped = [c for c in bld.columns if c not in ALKIS_SLIM_COLS]

slim = bld[ALKIS_SLIM_COLS].copy()
print()
print(f'  ..  {len(bld.columns)} columns -> {len(slim.columns)} '
      f'({len(dropped)} dropped)')
print(f'  ..  dropped: {", ".join(sorted(dropped))}')
require_unique(slim, 'alkis_id', 'slim buildings')

print()
print(slim.drop(columns='geometry').head(5).to_string(index=False))
print()
print('  the layer that goes into 04.2:')
for c in slim.columns:
    if c == 'geometry':
        continue
    nn = slim[c].notna().sum()
    print(f'      {c:<20} {100 * nn / len(slim):>6.2f} % filled   {slim[c].dtype}')

## 4. Attach the labels and the activities

Two reference tables, both keyed on the same `31001_1000` form as the layer, so
both are plain left joins:

| file | rows | gives |
|---|---|---|
| `building_function_codelist_de_en.csv` | 301 | `label_de`, `label_en` — what the code *means* |
| `alkis_building_activity_map.csv` | 280 | `activities` — what *happens* there |

The activity map is the consequential one: `activities` is what the
redistribution eventually weights, so a code missing from it yields a building
with no activity, which would drop out silently rather than raise anything.

**Measured coverage: 88 of 88 codes present in both.** Nothing falls out. The
check below asks the question in that direction anyway — *which of OUR codes are
missing?* — rather than the reassuring but useless "how much of the reference do
we use?".

The activity map was converted once from `alkis_building_activity_map.xlsx`
(kept beside it) so the pipeline needs no `openpyxl` and the table stays
greppable. Both CSVs carry a BOM, hence `utf-8-sig`: read as plain `utf-8` the
first column comes back with a `\ufeff` prefix and the join matches nothing.

The 14 atomic activities in use are `business, daycare, early_education,
education, errands, home, leisure, lessons, meetup, other, shopping, sports,
unspecified, work`. Note `unspecified` and `other` are real values in the
source, not placeholders — a building can legitimately end up with nothing
informative.

In [ ]:
require_file(BUILDING_FUNCTION_CODELIST_FILE, 'AdV function codelist')
require_file(ALKIS_ACTIVITY_MAP_FILE, 'ALKIS activity map')

codes = pd.read_csv(BUILDING_FUNCTION_CODELIST_FILE, encoding='utf-8-sig', dtype=str)
acts = pd.read_csv(ALKIS_ACTIVITY_MAP_FILE, encoding='utf-8-sig', dtype=str)

# Both are keyed tables, so a duplicate key would quietly multiply rows in the
# join. Asserted, not assumed.
require_unique(codes, 'function', 'codelist')
if 'gfk_code' not in acts.columns:
    raise AssertionError(f'activity map columns are {list(acts.columns)}, '
                         "expected a 'gfk_code' column - check the BOM")
acts = acts.rename(columns={'gfk_code': 'function'})
require_unique(acts, 'function', 'activity map')
print(f'  ..  codelist {len(codes)} codes | activity map {len(acts)} codes')

# --- coverage, asked in the direction that can hurt -------------------------
ours = (slim.groupby('function')
        .agg(n=('alkis_id', 'size'), vol=('volume_3d_m3', 'sum')))
for tbl, name in ((codes, 'codelist'), (acts, 'activity map')):
    missing = ours[~ours.index.isin(set(tbl['function']))]
    if len(missing):
        print(f'  !!  {len(missing)} of our {len(ours)} codes are ABSENT from the '
              f'{name}: {missing["n"].sum():,} buildings '
              f'({100 * missing["n"].sum() / len(slim):.2f} %), '
              f'{missing["vol"].sum() / 1e6:,.1f}M m3 '
              f'({100 * missing["vol"].sum() / slim["volume_3d_m3"].sum():.2f} % of volume)')
        print(missing.sort_values('n', ascending=False).head(10).to_string())
    else:
        print(f'  ok  all {len(ours)} of our codes are present in the {name}')

# --- join --------------------------------------------------------------------
before = len(slim)
slim = slim.merge(codes[['function', 'label_de', 'label_en']], on='function', how='left')
slim = slim.merge(acts[['function', 'activities']], on='function', how='left')
if len(slim) != before:
    raise AssertionError(f'the joins changed the row count {before:,} -> {len(slim):,} '
                         '- a reference table has duplicate keys')
print(f'  ok  joined, still {len(slim):,} rows')

for c in ('label_en', 'activities'):
    nn = slim[c].notna().sum()
    print(f'  ..  {c:<12} {100 * nn / len(slim):>6.2f} % filled')

# The activity list, exploded, so it can be counted per atomic activity.
slim['n_activities'] = (slim['activities'].fillna('')
                        .str.split(ALKIS_ACTIVITY_SEP)
                        .map(lambda xs: len([x for x in xs if x.strip()])))
print()
print('  ..  buildings per atomic activity:')
_ex = (slim[['alkis_id', 'volume_3d_m3', 'activities']]
       .assign(a=slim['activities'].fillna('').str.split(ALKIS_ACTIVITY_SEP))
       .explode('a'))
_ex['a'] = _ex['a'].str.strip()
_ex = _ex[_ex['a'] != '']
_t = _ex.groupby('a').agg(buildings=('alkis_id', 'size'),
                          volume_Mm3=('volume_3d_m3', lambda v: v.sum() / 1e6))
_t['pct_vol'] = (100 * _t['volume_Mm3'] / (slim['volume_3d_m3'].sum() / 1e6)).round(2)
print(_t.sort_values('buildings', ascending=False).round(1).to_string())
_none = int(slim['n_activities'].eq(0).sum())
print(f'\n  ..  buildings with NO activity at all: {_none:,} '
      f'({100 * _none / len(slim):.3f} %)')

## 5. The review table — which building types are worth keeping?

This is the decision material, not a decision. One row per `function` code with
everything needed to judge it: what it is, what activities it has been assigned,
how many buildings, how much of the region's volume, and a physical profile
(median area, median height, share of flat roofs).

**The case that needs your eye most:**

```
51009_1610   Überdachung / canopy   →   activities: work;leisure
             94,571 buildings, 10.9 % of the layer
```

A canopy is a roof over a petrol forecourt or a bus stop. Nothing happens
*inside* it, because it has no inside — yet it carries `work;leisure`, so in a
volume-proportional redistribution those 94,571 structures compete with real
buildings for worker and leisure demand.

It is not obviously wrong to keep them: a covered loading bay at a depot is
arguably part of a workplace. But it is a decision, and the numbers below plus
the QGIS layer are what it should be made from rather than by inheriting whatever
the old pipeline did.

The `median_height` and `pct_flat_roof` columns are there because they separate
"structure" from "building" physically: a canopy is low and flat, a workshop is
not.

In [ ]:
prof = slim.groupby(['function', 'label_de', 'label_en', 'activities'],
                    dropna=False).agg(
    n_buildings=('alkis_id', 'size'),
    volume_Mm3=('volume_3d_m3', lambda v: round(v.sum() / 1e6, 3)),
    median_area_m2=('area_m2', 'median'),
    median_height_m=('height_top_max_m', 'median'),
    median_volume_m3=('volume_3d_m3', 'median'),
    pct_flat_roof=('roof_shape', lambda s: round(100 * (s == 'PolyFlatRoof').mean(), 1)),
    pct_named=('name', lambda s: round(100 * s.notna().mean(), 2)),
).reset_index()
prof['pct_buildings'] = (100 * prof['n_buildings'] / len(slim)).round(3)
prof['pct_volume'] = (100 * prof['volume_Mm3'] /
                      (slim['volume_3d_m3'].sum() / 1e6)).round(3)
prof = prof.sort_values('n_buildings', ascending=False)

cols = ['function', 'label_en', 'activities', 'n_buildings', 'pct_buildings',
        'volume_Mm3', 'pct_volume', 'median_area_m2', 'median_height_m',
        'pct_flat_roof', 'pct_named']
print('the 20 codes with the most buildings:')
print(prof[cols].head(20).to_string(index=False))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
prof.to_csv(FUNCTION_REVIEW_CSV, index=False, encoding='utf-8-sig')
print(f'\n  ok  all {len(prof)} codes -> {FUNCTION_REVIEW_CSV.name}')

print()
print('  ..  the non-31001 classes, which are structures rather than buildings:')
_n31 = prof[~prof['function'].str.startswith('31001')]
print(_n31[cols].head(12).to_string(index=False))
print(f'\n      {len(_n31)} codes, {_n31["n_buildings"].sum():,} buildings '
      f'({100 * _n31["n_buildings"].sum() / len(slim):.2f} %), '
      f'{_n31["volume_Mm3"].sum():,.1f}M m3 '
      f'({100 * _n31["volume_Mm3"].sum() / (slim["volume_3d_m3"].sum() / 1e6):.2f} % of volume)')

## 6. The class-by-class review export

Every one of the 869,316 buildings, labelled, **including the ones section 7
drops** — so the decision can be made by looking rather than from counts alone.

GeoPackage rather than shapefile: `label_en` and `activities` are long strings
and shapefile truncates field names to 10 characters.

### Stepping through the classes

A **`.qml` style file** is written beside the layer, so QGIS loads it already
categorised by `function` with all 88 classes in the legend. Each legend entry
can be ticked on and off individually — that is the class-by-class review,
without typing a single filter expression.

Colours are grouped by what the code family means, so the map is readable before
you touch anything:

| family | colour |
|---|---|
| `31001_1xxx` residential | blues |
| `31001_2xxx` business, commerce | oranges and reds |
| `31001_3xxx` public purposes | greens |
| `51xxx` other structures | greys |

Legend entries read `31001_1000 · residential buildings · 327,264` so the label
carries the code, the meaning and the size without a lookup.

Three extra columns make manual filtering easy where you do want it:

* **`class_label`** — the same `code · meaning · count` string, if you would
  rather categorise on that than on the bare code
* **`aaa_class`** — `31001` or `51009` etc., for splitting buildings from
  structures in one filter
* **`kept`** — what the section 7 rule removes. Styling by it shows the drop
  directly.

### Worth looking at specifically

1. **`function = '51009_1610'`** — 94,571 canopies, median 10 m² and 3.9 m.
   Carports and forecourt roofs, tagged `work;leisure`.
2. **`function = '31001_2000' AND area_m2 < 50`** — the shed question. 369,675
   buildings are tagged "business or commerce" with a median of 28.7 m².
3. **`aaa_class <> '31001'`** — all 100,560 non-building structures at once.
4. **`kept = false`** — should be housing and nothing else.

In [ ]:
EXPERIMENTAL_DIR.mkdir(parents=True, exist_ok=True)

# The home-only flag is computed here so the export can show what section 7
# removes. The rule itself is explained there.
_acts = (slim['activities'].fillna('')
         .str.split(ALKIS_ACTIVITY_SEP)
         .map(lambda xs: frozenset(x.strip() for x in xs if x.strip())))
slim['home_only'] = _acts.map(
    lambda a: bool(a) and a <= ALKIS_HOME_ONLY_ACTIVITIES)
slim['kept'] = ~slim['home_only']

# Columns that make the review easy without writing filter expressions.
slim['aaa_class'] = slim['function'].str.split('_').str[0]
_n = slim['function'].value_counts()
slim['class_label'] = (
    slim['function'] + ' \u00b7 ' + slim['label_en'].fillna('?')
    + ' \u00b7 ' + slim['function'].map(_n).map('{:,}'.format))

inspect_cols = ['alkis_id', 'function', 'aaa_class', 'class_label',
                'label_de', 'label_en', 'activities', 'n_activities', 'kept',
                'area_m2', 'volume_3d_m3', 'volume_old_m3', 'volume_ratio',
                'height_top_max_m', 'roof_shape', 'name', 'address', 'city',
                'n_parts', 'geometry']
insp = slim[inspect_cols]

if LABELLED_INSPECT_FILE.exists():
    try:
        LABELLED_INSPECT_FILE.unlink()
    except PermissionError as e:
        raise RuntimeError(
            f'{LABELLED_INSPECT_FILE.name} is locked - close it in QGIS and rerun '
            f'this cell. Original error: {e}'
        ) from None

print(f'Writing {len(insp):,} rows x {len(insp.columns)} columns to '
      f'{LABELLED_INSPECT_FILE.name} ...', flush=True)
t0 = time.perf_counter()
insp.to_file(LABELLED_INSPECT_FILE, layer='buildings', driver='GPKG')
print(f'  ok  {LABELLED_INSPECT_FILE.stat().st_size / 1e6:,.1f} MB  '
      f'[{time.perf_counter() - t0:,.0f}s]')

chk = gpd.read_file(LABELLED_INSPECT_FILE, layer='buildings', rows=4)
print(f'  ok  read back: CRS {chk.crs}, {len(chk.columns)} columns')
print(chk[['function', 'label_en', 'activities', 'kept', 'area_m2',
           'height_top_max_m']].to_string(index=False))

In [ ]:
# --- write a QGIS style so the layer opens already categorised --------------
# Hand-written QML rather than exported from QGIS, because QGIS is not scriptable
# from here. Uses the <prop k=.../> form, which QGIS 3.x still accepts, and is
# belt-and-braces: if a QGIS version refuses the file, `class_label` still gives
# a three-click path to the same categorised view.
#
# Colours carry meaning rather than being arbitrary: the family a code belongs to
# decides the hue, so the map is readable before any styling is touched.
_FAMILY_RAMPS = {
    '1': [(31, 120, 180), (66, 146, 198), (107, 174, 214), (158, 202, 225),
          (198, 219, 239), (222, 235, 247)],                      # residential: blues
    '2': [(227, 74, 51), (239, 101, 72), (252, 141, 89), (253, 187, 132),
          (253, 212, 158), (254, 232, 200)],                      # business: oranges/reds
    '3': [(35, 132, 67), (65, 171, 93), (120, 198, 121), (173, 221, 142),
          (217, 240, 163), (247, 252, 185)],                      # public: greens
}
_GREYS = [(82, 82, 82), (115, 115, 115), (150, 150, 150), (189, 189, 189),
          (217, 217, 217)]


def _colour(code, i):
    aaa, gfk = code.split('_', 1)
    if aaa != '31001':
        return _GREYS[i % len(_GREYS)]
    ramp = _FAMILY_RAMPS.get(gfk[0], _GREYS)
    return ramp[i % len(ramp)]


def _esc(t):
    return (str(t).replace('&', '&amp;').replace('<', '&lt;')
            .replace('>', '&gt;').replace('"', '&quot;'))


# Biggest classes first, so the legend reads in order of importance.
order = slim['function'].value_counts().index.tolist()
labels = (slim.drop_duplicates('function').set_index('function')['class_label'])

cats, syms, seen = [], [], {}
for n, code in enumerate(order):
    fam = code.split('_', 1)[1][0] if code.startswith('31001') else 'x'
    seen[fam] = seen.get(fam, 0)
    r, g, b = _colour(code, seen[fam])
    seen[fam] += 1
    cats.append(f'   <category value="{_esc(code)}" symbol="{n}" '
                f'label="{_esc(labels.get(code, code))}" render="true"/>')
    syms.append(
        f'   <symbol type="fill" name="{n}" alpha="1" clip_to_extent="1" force_rhr="0">\n'
        f'    <layer class="SimpleFill" enabled="1" pass="0" locked="0">\n'
        f'     <prop k="color" v="{r},{g},{b},255"/>\n'
        f'     <prop k="style" v="solid"/>\n'
        f'     <prop k="outline_color" v="35,35,35,180"/>\n'
        f'     <prop k="outline_style" v="solid"/>\n'
        f'     <prop k="outline_width" v="0.04"/>\n'
        f'     <prop k="outline_width_unit" v="MM"/>\n'
        f'     <prop k="joinstyle" v="bevel"/>\n'
        f'    </layer>\n'
        f'   </symbol>')

qml = (
    "<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>\n"
    '<qgis version="3.28.0" styleCategories="Symbology">\n'
    ' <renderer-v2 type="categorizedSymbol" attr="function" forceraster="0"\n'
    '              symbollevels="0" enableorderby="0" referencescale="-1">\n'
    '  <categories>\n' + '\n'.join(cats) + '\n  </categories>\n'
    '  <symbols>\n' + '\n'.join(syms) + '\n  </symbols>\n'
    ' </renderer-v2>\n'
    ' <blendMode>0</blendMode>\n'
    '</qgis>\n'
)

qml_path = LABELLED_INSPECT_FILE.with_suffix('.qml')
qml_path.write_text(qml, encoding='utf-8')

# Parse it back - a malformed QML fails silently in QGIS, which is worse than
# an error here.
import xml.etree.ElementTree as _ET
_root = _ET.fromstring(qml)
_ncat = len(_root.findall('.//category'))
_nsym = len(_root.findall('.//symbol'))
if _ncat != len(order) or _nsym != len(order):
    raise AssertionError(f'QML has {_ncat} categories and {_nsym} symbols, '
                         f'expected {len(order)} of each')
print(f'  ok  {qml_path.name}: {_ncat} categories, valid XML, '
      f'{qml_path.stat().st_size / 1e3:,.0f} KB')
print()
print('  ..  in QGIS: add 04_buildings_labelled.gpkg - the .qml loads with it,')
print('      and every class appears in the legend with its own tick box.')
print()
print('  ..  the legend, biggest first:')
for code in order[:12]:
    print(f'        {labels.get(code, code)}')
print(f'        ... and {len(order) - 12} more')

## 7. Drop the buildings that are only somewhere to live

A capacity model places demand that happens *at* a building — work, shopping,
errands, education, leisure. A building whose entire activity set is
`{home, meetup}` hosts none of it, so it is removed.

**Expressed as a rule over `activities`, not as a code list.** That is what
keeps the interesting case: a flat with a shop on the ground floor carries
`home;meetup;work;shopping;business;errands`, which is not a subset of
`{home, meetup}`, so it survives without anyone maintaining an exception list.

The rule's expected result is declared in `config.py` and **asserted against the
data** — if the computed set differs, the activity map or the source release has
changed and the reasoning needs revisiting rather than the list being quietly
edited.

### What this costs, decided deliberately

**`meetup` is a redistribution target.** The original pipeline maps MiD
`meetup → Leisure`, and dropping `31001_1000` removes **302.5M m³** of it. So
visiting-friends trips have nowhere to land, and Leisure demand falls entirely
on pubs, sports halls and cinemas.

Accepted: home visits are out of scope for a capacity model. Worth revisiting
here if the Leisure totals later look implausibly concentrated on venues.

In [ ]:
# `home_only` and `kept` were computed in section 6 so the map could show them.
by_code = (slim[slim['home_only']]
           .groupby(['function', 'label_en', 'activities'])
           .agg(n=('alkis_id', 'size'), vol=('volume_3d_m3', 'sum')))

found = set(slim.loc[slim['home_only'], 'function'].unique())
expected = set(ALKIS_EXPECTED_HOME_ONLY)
if found != expected:
    raise AssertionError(
        'the home-only rule selects a different set of codes than '
        'ALKIS_EXPECTED_HOME_ONLY in config.py.\n'
        f'  only in data   : {sorted(found - expected)}\n'
        f'  only in config : {sorted(expected - found)}\n'
        'The activity map or the source release has changed - review the '
        'reasoning in config.py before editing the list.'
    )
print(f'  ok  {len(found)} home-only codes, matching config.py exactly')
print()
print('DROP - activities are a subset of '
      f'{{{", ".join(sorted(ALKIS_HOME_ONLY_ACTIVITIES))}}}:')
for (fn, lab, act), row in by_code.iterrows():
    print(f'  {fn:<12} {lab[:44]:<44} {act:<12} {int(row["n"]):>8,} bld  '
          f'{row["vol"] / 1e6:>7.1f}M m3')
    print(f'  {"":<12} {ALKIS_EXPECTED_HOME_ONLY[fn]}')

n_before = len(slim)
v_before = slim['volume_3d_m3'].sum()
enriched = slim[slim['kept']].drop(columns=['home_only', 'kept']).copy()
enriched = enriched.reset_index(drop=True)

print()
print(f'  ..  buildings : {n_before:,} -> {len(enriched):,} '
      f'({n_before - len(enriched):,} dropped, '
      f'{100 * (n_before - len(enriched)) / n_before:.2f} %)')
print(f'  ..  volume    : {v_before / 1e6:,.1f}M -> '
      f'{enriched["volume_3d_m3"].sum() / 1e6:,.1f}M m3 '
      f'({100 * (1 - enriched["volume_3d_m3"].sum() / v_before):.2f} % removed)')
require_unique(enriched, 'alkis_id', 'enriched buildings')

# Nothing carrying a non-home activity may have been removed. This is the check
# that the "flat with a shop downstairs" case actually survived.
_gone = slim[~slim['kept']]
_leak = _gone[_gone['activities'].fillna('').str.contains(
    'work|business|shopping|errands|education|lessons|sports|daycare|leisure',
    regex=True)]
if len(_leak):
    raise AssertionError(
        f'{len(_leak):,} dropped buildings carry a non-home activity: '
        f'{_leak["function"].unique().tolist()}'
    )
print('  ok  nothing dropped carries work, shopping, errands, education or leisure')

print()
print('  ..  mixed residential retained:')
_mix = enriched[enriched['activities'].fillna('').str.contains('home')]
_t = (_mix.groupby(['function', 'label_en'])
      .agg(n=('alkis_id', 'size'), vol_Mm3=('volume_3d_m3', lambda v: round(v.sum() / 1e6, 3)))
      .sort_values('n', ascending=False))
print(_t.to_string())
print(f'      {len(_mix):,} buildings kept because they also host non-home activity')

print()
print('  ..  what the enriched layer now looks like, by activity:')
_ex = (enriched[['alkis_id', 'volume_3d_m3']]
       .assign(a=enriched['activities'].fillna('').str.split(ALKIS_ACTIVITY_SEP))
       .explode('a'))
_ex['a'] = _ex['a'].str.strip()
_ex = _ex[_ex['a'] != '']
_t2 = _ex.groupby('a').agg(buildings=('alkis_id', 'size'),
                           volume_Mm3=('volume_3d_m3', lambda v: round(v.sum() / 1e6, 1)))
_t2['pct_vol'] = (100 * _t2['volume_Mm3'] /
                  (enriched['volume_3d_m3'].sum() / 1e6)).round(1)
print(_t2.sort_values('buildings', ascending=False).to_string())

## 8. Where this leaves us

`enriched` holds **537,855 buildings** — every one of them a place where
something other than living happens — with `label_de`, `label_en`, `activities`
and `n_activities` attached. Still in memory; the layer is written once the POI
join is in.

Two review files are on disk:

* **`04_function_activity_review.csv`** — all 88 codes with labels, activities,
  counts, volume share and a physical profile
* **`04_buildings_labelled.gpkg`** — all 869,316 buildings including the dropped
  ones, with a `kept` flag so the drop is visible on the map

### What the review table exposed, still undecided

* **`51009_1610` canopies — 94,571 buildings, tagged `work;leisure`.** Median
  10 m² and 3.9 m tall: carports and forecourt roofs. Nothing happens *inside* a
  canopy because it has no inside, yet they will compete for worker and leisure
  demand. They are only 1.5 % of volume, so the damage is bounded — but it is
  still a decision nobody has made.
* **`31001_2000` "buildings for business or commerce" — 369,675 buildings,
  42.5 % of the layer, tagged `work;business`.** Median area 28.7 m², median
  height 2.9 m, 74 % flat-roofed. That is a garden shed or a garage, not a
  commercial premises. It is 15.2 % of volume against 42.5 % of count, so volume
  weighting limits the harm, but 370,000 sheds carrying `work` is the largest
  single question left in this layer.
* Masts, chimneys, wind turbines and solar panels all carry `work` too.

A size threshold would address most of it at once, which is the decision step 03
deliberately deferred and this table is the evidence for.

### Next: 04.4, the OSM POI join

`01_all_pois.gpkg` joined by `poi_role` — points by containment, footprints 1:1,
sites onto every contained building. That is where the nested shopping-centre
units come in.

**Caveat carried forward:** notebook 01 has never been tested, 04.4 depends
entirely on its POI layer, and its `poi_role` has no concept of a POI nested
inside another POI — which is exactly what a shopping centre is. Worth verifying
before the join is built on top of it.